In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler

# 设置随机种子以保证结果可复现
torch.manual_seed(42)
np.random.seed(42)

# --- 2.2 编程题 & 3.2 编程题 ---
# 1. 手动实现 MLP (含 ReLU, Softmax, SGD)
class SimpleMLP:
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_prob=0.0, weight_decay=0.0):
        self.dropout_prob = dropout_prob
        self.weight_decay = weight_decay
        self.is_training = True
        
        # 1. 参数初始化 (正态分布)
        # 隐藏层
        self.W1 = torch.randn(input_dim, hidden_dim) * 0.01
        self.b1 = torch.zeros(hidden_dim)
        # 输出层
        self.W2 = torch.randn(hidden_dim, output_dim) * 0.01
        self.b2 = torch.zeros(output_dim)
        
        # 梯度存储
        self.dW1, self.db1, self.dW2, self.db2 = None, None, None, None

    # 2. ReLU 激活函数
    def relu(self, x):
        return torch.max(x, torch.zeros_like(x))

    # 3. Dropout 从零实现 (3.2要求)
    def dropout_layer(self, X):
        if not self.is_training or self.dropout_prob == 0.0:
            return X
        # 生成随机掩码，大于 dropout_prob 的位置保留
        mask = torch.rand(X.shape) > self.dropout_prob
        # 缩放：除以保留概率 (Inverted Dropout)
        scale = 1.0 / (1.0 - self.dropout_prob)
        return mask * X * scale

    # 前向传播
    def forward(self, X):
        # 隐藏层: Wx + b -> ReLU
        self.h1 = torch.matmul(X, self.W1) + self.b1
        self.a1 = self.relu(self.h1)
        
        # Dropout (在隐藏层后)
        self.a1_drop = self.dropout_layer(self.a1)
        
        # 输出层: Wx + b
        self.logits = torch.matmul(self.a1_drop, self.W2) + self.b2
        return self.logits

    # 4. 带 Softmax 的交叉熵损失 (手动实现)
    def compute_loss(self, logits, y_true):
        # 数值稳定性技巧：减去最大值
        logits = logits - torch.max(logits, dim=1, keepdim=True)[0]
        
        # Softmax
        exp_logits = torch.exp(logits)
        self.probs = exp_logits / torch.sum(exp_logits, dim=1, keepdim=True)
        
        # 交叉熵 Loss: -sum(y_true * log(p))
        # 使用高级API的gather来获取正确类别的概率，或者手动构建one-hot
        m = y_true.shape[0]
        correct_logprobs = -torch.log(self.probs[range(m), y_true] + 1e-8) # 加epsilon防溢出
        data_loss = torch.sum(correct_logprobs) / m
        
        # 加入 L2 正则化 (权重衰减) (3.2要求)
        reg_loss = 0.5 * self.weight_decay * (torch.sum(self.W1**2) + torch.sum(self.W2**2))
        return data_loss + reg_loss

    # 反向传播
    def backward(self, X, y_true):
        m = X.shape[0]
        
        # 1. 计算 Loss 对 logits 的梯度 (Softmax + Cross-Entropy 合并求导)
        # dL/dz = probs - one_hot(y)
        dscores = self.probs.clone()
        dscores[range(m), y_true] -= 1
        dscores = dscores / m # 平均化
        
        # 2. 反向传播到输出层 W2, b2
        # dL/dW2 = a1^T * dscores
        self.dW2 = torch.matmul(self.a1_drop.t(), dscores)
        self.db2 = torch.sum(dscores, dim=0)
        
        # 3. 反向传播通过 Dropout
        dhidden = torch.matmul(dscores, self.W2.t())
        # 反向传播时也需要应用 Dropout Mask
        if self.is_training and self.dropout_prob > 0.0:
            mask = torch.rand(self.a1.shape) > self.dropout_prob
            scale = 1.0 / (1.0 - self.dropout_prob)
            dhidden = dhidden * mask * scale
            
        # 4. 反向传播通过 ReLU
        dhidden[self.h1 <= 0] = 0
        
        # 5. 反向传播到隐藏层 W1, b1
        self.dW1 = torch.matmul(X.t(), dhidden)
        self.db1 = torch.sum(dhidden, dim=0)

    # 5. 手动 SGD 更新 (含权重衰减)
    def step(self, lr):
        # 权重衰减: 梯度更新前，权重先乘以 (1 - lr*lambda)
        # 或者在梯度上加上 lambda*W (效果等同)
        if self.weight_decay > 0:
            self.dW2 += self.weight_decay * self.W2
            self.dW1 += self.weight_decay * self.W1
            
        # 执行更新
        self.W1 -= lr * self.dW1
        self.b1 -= lr * self.db1
        self.W2 -= lr * self.dW2
        self.b2 -= lr * self.db2

In [3]:
# --- 4.2 编程题 ---
def experiment_gradient_flow():
    # 构建 20 层网络
    layers = []
    input_dim = 256
    hidden_dim = 256
    num_layers = 20
    
    # 尝试不同的组合
    # Case 1: Sigmoid + 普通高斯 (梯度消失)
    # Case 2: ReLU + 大初值 (梯度爆炸)
    # Case 3: ReLU + Xavier (稳定)
    
    # 这里演示 Case 3: Xavier + ReLU
    for i in range(num_layers):
        lin = nn.Linear(hidden_dim, hidden_dim) if i > 0 else nn.Linear(input_dim, hidden_dim)
        # 推荐使用 Xavier Uniform
        nn.init.xavier_uniform_(lin.weight)
        # 偏置通常初始化为0
        nn.init.zeros_(lin.bias)
        layers.append(lin)
        layers.append(nn.ReLU()) # 或者 nn.LeakyReLU()
    
    net = nn.Sequential(*layers)
    
    # 模拟输入数据
    x = torch.randn(64, input_dim)
    
    # 注册钩子来获取梯度 (或者手动记录)
    # 简单方法：前向+反向后查看
    output = net(x)
    # 假设一个虚拟 Loss
    loss = output.sum()
    loss.backward()
    
    # 打印第一层和最后一层的梯度范数
    first_layer_grad = net[0].weight.grad
    last_layer_grad = net[-2].weight.grad # -2是因为最后一个是ReLU，没有参数
    
    print(f"第一层梯度范数: {torch.norm(first_layer_grad):.6f}")
    print(f"最后一层梯度范数: {torch.norm(last_layer_grad):.6f}")
    # 理想状态：数值稳定在 [1e-6, 1e3] 区间内

In [5]:
# --- 5.2 编程题 ---
def experiment_covariate_shift():
    # 1. 构造数据 (5.2要求)
    # 训练集 P: N(-1, 1)
    X_train = np.random.normal(-1, 1, 1000).reshape(-1, 1)
    y_train = 2 * X_train + np.random.normal(0, 0.1, X_train.shape) # y = 2x + noise
    
    # 测试集 Q: N(2, 1) -> 发生了协变量偏移
    X_test = np.random.normal(2, 1, 500).reshape(-1, 1)
    y_test = 2 * X_test + np.random.normal(0, 0.1, X_test.shape)
    
    # 转换为 Tensor
    X_train_t = torch.FloatTensor(X_train)
    X_test_t = torch.FloatTensor(X_test)
    
    # 2. 基线模型 (直接训练)
    # 这里简化为线性回归手动实现或使用 sklearn
    # 假设我们有一个训练好的模型，直接评估
    # baseline_mse = ...
    
    # 3. 偏移校正 (重要性采样)
    # (a) 训练判别器: 预测样本属于测试集的概率 P(test|x)
    
    # 标记数据: 训练集为 0, 测试集为 1
    X_all = torch.cat([X_train_t, X_test_t], dim=0)
    y_domain = torch.cat([torch.zeros(X_train_t.shape[0]), torch.ones(X_test_t.shape[0])], dim=0)
    
    # 简单的逻辑回归判别器
    classifier = nn.Sequential(
        nn.Linear(1, 10),
        nn.ReLU(),
        nn.Linear(10, 1),
        nn.Sigmoid()
    )
    
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)
    criterion = nn.BCELoss()
    
    # 训练判别器
    for epoch in range(100):
        pred = classifier(X_all).squeeze()
        loss = criterion(pred, y_domain)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # (b) 计算权重 w_i
    # P(test|x) 是判别器的输出
    with torch.no_grad():
        p_test_given_x = classifier(X_train_t).squeeze() # 训练集样本被判别为测试集的概率
        p_train_given_x = 1 - p_test_given_x # 或者直接用判别器输出的反面
        
        # 计算重要性权重: w_i propto P(test|x) / P(train|x)
        # 防止除以0
        weights = (p_test_given_x / (p_train_given_x + 1e-8)).numpy()
    
    # 4. 加权模型训练 (加权最小二乘)
    # 这里演示如何应用权重 (伪代码逻辑)
    # model.fit(X_train, y_train, sample_weight=weights)
    # 然后在 X_test 上评估 MSE，对比校正前后的效果
    
    return weights, X_train, y_train, X_test, y_test

# 调用函数获取权重
# weights, ... = experiment_covariate_shift()